# AaharAI Interactive AI Research Environment
Welcome to the Interactive Research notebook. The data loading and visualization environments are ready for your inspection.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import joblib
import os

# Set up SQL connection directly to backend data
db_path = '../backend/data/aahar.db'
conn = sqlite3.connect(db_path)

## 1. Data Loading
Load distributions into a Pandas DataFrame.

In [3]:
query = "SELECT * FROM distributions WHERE status = 'completed'"
df = pd.read_sql_query(query, conn)
df['distributed_at'] = pd.to_datetime(df['distributed_at'], format='mixed')
print(f"Loaded {len(df)} historical distributions.")
df.head()

ValueError: Mixed timezones detected. Pass utc=True in to_datetime or tz='UTC' in DatetimeIndex to convert to a common timezone.

## 2. Feature Engineering
Extract hour, day_of_week, and is_weekend from the timestamps.

In [ ]:
df['hour'] = df['distributed_at'].dt.hour
df['day_of_week'] = df['distributed_at'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

df[['distributed_at', 'hour', 'day_of_week', 'is_weekend']].head()

## 3. Exploratory Data Analysis (EDA)
Show demand distribution and correlation heatmap. **(Please review visually before continuing below)**

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['hour'], bins=24, kde=True)
plt.title('Demand Distribution by Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Transaction Count')
plt.show()

plt.figure(figsize=(10, 6))
# Select only numeric columns for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()

## 4. Interactive XGBoost
Set up training cell for XGBRegressor with manual Hyperparameter Tuning.

In [ ]:
# Organize data into daily demand matching commodities and vendors
daily_demand = df.groupby([df['distributed_at'].dt.date, 'commodity_id', 'vendor_id'])['quantity'].sum().reset_index()
daily_demand['distributed_at'] = pd.to_datetime(daily_demand['distributed_at'])
daily_demand['day_of_week'] = daily_demand['distributed_at'].dt.dayofweek
daily_demand['is_weekend'] = daily_demand['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

X = daily_demand[['commodity_id', 'vendor_id', 'day_of_week', 'is_weekend']]
y = daily_demand['quantity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### HYPERPARAMETER TUNING SECTION ###
params = {
    'max_depth': 4,
    'learning_rate': 0.1,
    'n_estimators': 100
}
######################################

model_xgb = XGBRegressor(**params, random_state=42)
model_xgb.fit(X_train, y_train)

preds = model_xgb.predict(X_test)
mse = mean_squared_error(y_test, preds)
print(f'Model MSE: {mse:.4f}')

## 5. Visual Validation
Plot Feature Importance and Actual vs. Predicted graphs.

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.barplot(x=model_xgb.feature_importances_, y=X.columns)
plt.title('XGBoost Feature Importance')

plt.subplot(1, 2, 2)
plt.scatter(y_test, preds, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Quantity')
plt.ylabel('Predicted Quantity')
plt.title('Actual vs Predicted Demand')

plt.tight_layout()
plt.show()

## 6. Anomaly Research (Isolation Forest)
Plot a 'Contour Map' of anomalies to visually pinpoint fraudulent transactions.

In [ ]:
# Detect anomalies using transaction quantity and time of day
X_anomaly = df[['quantity', 'hour']].copy()

iso_forest = IsolationForest(contamination=0.05, random_state=42)
df['anomaly'] = iso_forest.fit_predict(X_anomaly)

# Create meshgrid to draw contour boundaries
xx, yy = np.meshgrid(np.linspace(X_anomaly['quantity'].min()-1, X_anomaly['quantity'].max()+1, 50),
                     np.linspace(X_anomaly['hour'].min()-1, X_anomaly['hour'].max()+1, 50))
Z = iso_forest.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, Z, cmap=plt.cm.Blues_r)
# Plot points (White = Normal, Red = Anomaly/Fraud risk)
colors = df['anomaly'].map({1: 'white', -1: 'red'})
plt.scatter(X_anomaly['quantity'], X_anomaly['hour'], 
            c=colors, edgecolor='k', s=30, alpha=0.8)
plt.xlabel('Quantity Distributed')
plt.ylabel('Hour of Day')
plt.title('Isolation Forest Anomaly Contours')
plt.show()

## 7. Export Logic
Export your tuned model parameters into a production-ready `train.py` script.

In [ ]:
train_py_content = """# Production Model Training Script (train.py)
import sqlite3
import pandas as pd
from xgboost import XGBRegressor
from sklearn.ensemble import IsolationForest
import joblib
import os

def extract_features(df):
    df['distributed_at'] = pd.to_datetime(df['distributed_at'])
    df['hour'] = df['distributed_at'].dt.hour
    df['day_of_week'] = df['distributed_at'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
    return df

def train_models():
    print("Loading dataset...")
    conn = sqlite3.connect('../backend/data/aahar.db')
    df = pd.read_sql_query("SELECT * FROM distributions WHERE status = 'completed'", conn)
    df = extract_features(df)
    
    print("Training XGBoost Demand Predictor...")
    daily_demand = df.groupby([df['distributed_at'].dt.date, 'commodity_id', 'vendor_id'])['quantity'].sum().reset_index()
    daily_demand = extract_features(daily_demand)
    X = daily_demand[['commodity_id', 'vendor_id', 'day_of_week', 'is_weekend']]
    y = daily_demand['quantity']
    
    best_params = {params} # Fetched directly from your notebook!
    model_xgb = XGBRegressor(**best_params, random_state=42)
    model_xgb.fit(X, y)
    
    os.makedirs('models', exist_ok=True)
    joblib.dump(model_xgb, 'models/xgboost_demand.pkl')
    
    print("Training Isolation Forest Anomaly Detector...")
    X_anomaly = df[['quantity', 'hour']].copy()
    iso_forest = IsolationForest(contamination=0.05, random_state=42)
    iso_forest.fit(X_anomaly)
    joblib.dump(iso_forest, 'models/isolation_forest.pkl')
    
    print("\nSuccess! Models saved to ai_engine/models/")

if __name__ == '__main__':
    train_models()
"""

# Ensure the dictionary params is serialized to a string in the script
script = train_py_content.replace('{params}', str(params))

with open('train.py', 'w') as f:
    f.write(script)
    
print("Success! `train.py` has been generated in the ai_engine/ directory.")